In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Change to your project directory (update this path to match your Drive structure)
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    # Install required packages
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

# CIFAR-10 Rotation-Based Clustering Analysis (Weight Updates)

This notebook divides CIFAR-10 dataset into 4 rotation-based clusters (0°, 90°, 180°, 270°), assigns them to 50 clients, and performs clustering analysis on the **model weight updates** from the second epoch using the last layer.

**Key Difference:** Instead of using raw gradients, this version uses weight updates (Δw = w_after - w_before), which is more aligned with federated learning algorithms like FedAvg.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine
import copy
import random
from collections import defaultdict

from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

In [ ]:
CONFIG = {
    'num_rotation_clusters': 4,  # 0°, 90°, 180°, 270°
    'num_clients': 50,
    'seed': 42,
    'model_name': 'resnet18',
    'pretrained': False,
    'batch_size': 64,
    'lr': 0.01,
    'warmup_epochs': 2,
}

# Set random seeds
set_seed(CONFIG['seed'])

## Step 1: Create Rotation-Based Dataset

We'll create a custom dataset that applies rotations to CIFAR-10 images and organize them into 4 clusters.

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """Custom dataset that applies rotation to CIFAR-10 images"""
    def __init__(self, cifar_dataset, rotation_angle, transform=None):
        self.cifar_dataset = cifar_dataset
        self.rotation_angle = rotation_angle
        self.transform = transform
        self.base_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        
    def __len__(self):
        return len(self.cifar_dataset)
    
    def __getitem__(self, idx):
        image, label = self.cifar_dataset[idx]
        
        # Apply rotation
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        else:
            image = self.base_transform(image)
        
        return image, label

# Load CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## Step 2: Assign Clients to Rotation Clusters

Divide the 50 clients evenly across 4 rotation clusters (0°, 90°, 180°, 270°).

In [ ]:
# Define rotation angles
rotation_angles = [0, 90, 180, 270]

# Assign clients to rotation clusters
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    # Each rotation cluster gets equal number of clients
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    # Handle remainder for last cluster
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"Client to rotation mapping (total {len(client_rotation_labels)} clients):")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

## Step 3: Create Client Data Subsets

Distribute training data among clients with their assigned rotations.

In [ ]:
# Split dataset among clients
samples_per_client = len(train_dataset) // CONFIG['num_clients']
all_indices = list(range(len(train_dataset)))
random.shuffle(all_indices)

train_subsets = []
client_sizes = []

for client_idx in range(CONFIG['num_clients']):
    start_idx = client_idx * samples_per_client
    end_idx = start_idx + samples_per_client
    
    # Handle remainder for last client
    if client_idx == CONFIG['num_clients'] - 1:
        end_idx = len(train_dataset)
    
    client_indices = all_indices[start_idx:end_idx]
    
    # Create rotated dataset for this client
    rotation_angle = client_rotation_labels[client_idx]
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset, rotation_angle)
    client_subset = Subset(rotated_dataset, client_indices)
    
    train_subsets.append(client_subset)
    client_sizes.append(len(client_subset))

# Visualization
plt.figure(figsize=(12, 5))

# Plot 1: Client sizes
plt.subplot(1, 2, 1)
plt.bar(range(CONFIG['num_clients']), client_sizes)
plt.xlabel('Client ID')
plt.ylabel('Number of Samples')
plt.title('Data Distribution Across Clients')
plt.grid(True, alpha=0.3)

# Plot 2: Rotation distribution
plt.subplot(1, 2, 2)
rotation_counts = [client_rotation_labels.count(angle) for angle in rotation_angles]
plt.bar([f"{angle}°" for angle in rotation_angles], rotation_counts)
plt.xlabel('Rotation Angle')
plt.ylabel('Number of Clients')
plt.title('Clients per Rotation Cluster')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"Average samples per client: {np.mean(client_sizes):.1f}")
print(f"Std dev: {np.std(client_sizes):.1f}")

## Step 4: Initialize Global Model

Initialize the global model that will be used as the starting point for each client.

In [ ]:
# Initialize global model
global_model = get_model(
    model_name=CONFIG['model_name'],
    pretrained=CONFIG['pretrained']
).to(device)

# Prepare test dataset
test_rotated = RotatedCIFAR10Dataset(test_dataset, 0)  # Use 0° for test set
test_subset = Subset(test_rotated, list(range(len(test_dataset))))
test_loader = DataLoader(test_subset, batch_size=CONFIG['batch_size'], shuffle=False)

print("Global model initialized successfully")
print(f"Model architecture: {CONFIG['model_name']}")

# Get layer names
layer_names = [name for name, _ in global_model.named_parameters()]
print(f"\nTotal layers: {len(layer_names)}")
print(f"Last 5 layers: {layer_names[-5:]}")

## Step 5: Run Local Training and Collect Weight Updates

Train each client locally for 2 epochs and collect the weight updates (Δw = w_after - w_before) for each epoch.

In [ ]:
# Storage for weight updates
client_weight_updates = {}  # client_idx -> [epoch_updates]
client_layer_updates = {}   # client_idx -> [epoch_layer_updates]

criterion = nn.CrossEntropyLoss()

print("Starting local training and collecting weight updates...")

for client_idx in range(CONFIG['num_clients']):
    # Initialize storage for this client
    client_weight_updates[client_idx] = []
    client_layer_updates[client_idx] = []
    
    # Get client's data loader
    train_loader = DataLoader(
        train_subsets[client_idx], 
        batch_size=CONFIG['batch_size'], 
        shuffle=True
    )
    
    # Create local model copy from global model
    local_model = copy.deepcopy(global_model)
    optimizer = optim.SGD(local_model.parameters(), lr=CONFIG['lr'], momentum=0.9)
    
    # Train for specified epochs
    local_model.train()
    for epoch in range(CONFIG['warmup_epochs']):
        # Store weights BEFORE this epoch
        weights_before = {name: param.clone().detach().cpu() 
                         for name, param in local_model.named_parameters()}
        
        # Train for one epoch
        epoch_loss = 0.0
        num_batches = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = local_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        # Store weights AFTER this epoch
        weights_after = {name: param.clone().detach().cpu() 
                        for name, param in local_model.named_parameters()}
        
        # Compute weight updates (Δw = w_after - w_before)
        layer_updates = {}
        flat_updates = []
        
        for name in weights_before.keys():
            update = weights_after[name] - weights_before[name]
            layer_updates[name] = update.numpy()
            flat_updates.append(update.view(-1))
        
        # Store flattened update
        flat_update = torch.cat(flat_updates).numpy()
        client_weight_updates[client_idx].append(flat_update)
        
        # Store layer-wise updates
        client_layer_updates[client_idx].append(layer_updates)
        
        avg_loss = epoch_loss / num_batches
    
    if (client_idx + 1) % 10 == 0:
        print(f"Completed {client_idx + 1}/{CONFIG['num_clients']} clients")

print("\nWeight update collection complete!")
print(f"Clients processed: {len(client_weight_updates)}")
print(f"Epochs per client: {len(client_weight_updates[0])}")
print(f"Update vector size: {len(client_weight_updates[0][0]):,} parameters")

## Step 6: Extract Last Layer Updates from Second Epoch

Extract the weight updates from the last layer (FC layer) for the second epoch.

In [ ]:
# Extract updates from the last layer (fc layer) for the second epoch (epoch index 1)
epoch_idx = 1  # Second epoch (0-indexed)

# Find FC layers
sample_client = 0
sample_layer_updates = client_layer_updates[sample_client][0]
layer_names = list(sample_layer_updates.keys())
print(f"Available layers: {layer_names[-5:]}")  # Show last 5 layers

# Find last layer (fc layer)
fc_layers = [name for name in layer_names if 'fc' in name]
print(f"FC layers: {fc_layers}")

# Extract updates from fc layers
last_layer_updates = []
for client_idx in range(CONFIG['num_clients']):
    if client_idx in client_layer_updates and len(client_layer_updates[client_idx]) > epoch_idx:
        epoch_updates = client_layer_updates[client_idx][epoch_idx]
        
        # Concatenate fc.weight and fc.bias updates
        fc_updates = []
        for layer_name in fc_layers:
            if layer_name in epoch_updates:
                fc_updates.append(epoch_updates[layer_name].flatten())
        
        if fc_updates:
            combined_fc_update = np.concatenate(fc_updates)
            last_layer_updates.append(combined_fc_update)
        else:
            print(f"Warning: No FC updates found for client {client_idx}")
    else:
        print(f"Warning: Client {client_idx} has no updates for epoch {epoch_idx}")

last_layer_updates = np.array(last_layer_updates)
print(f"\nExtracted weight updates shape: {last_layer_updates.shape}")
print(f"Number of clients with updates: {len(last_layer_updates)}")

# Statistics about the updates
print(f"\nUpdate statistics:")
print(f"Mean absolute update: {np.mean(np.abs(last_layer_updates)):.6f}")
print(f"Std of updates: {np.std(last_layer_updates):.6f}")
print(f"Max absolute update: {np.max(np.abs(last_layer_updates)):.6f}")

## Step 7: Perform K-Means Clustering

Perform K-Means clustering on the weight updates with k=4 to match the rotation clusters.

In [ ]:
# Perform K-Means clustering with k=4 (matching rotation clusters)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=CONFIG['seed'], n_init=10)
predicted_clusters = kmeans.fit_predict(last_layer_updates)

# Calculate clustering metrics
silhouette = silhouette_score(last_layer_updates, predicted_clusters)

print(f"K-Means Clustering Results (k={n_clusters}):")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"\nCluster sizes:")
for i in range(n_clusters):
    count = np.sum(predicted_clusters == i)
    print(f"  Cluster {i}: {count} clients")

## Step 8: Compare with True Rotation Clusters

Analyze how well the weight update-based clustering aligns with the actual rotation-based clusters.

In [ ]:
# Map rotation angles to cluster IDs
rotation_to_id = {0: 0, 90: 1, 180: 2, 270: 3}
true_clusters = np.array([rotation_to_id[angle] for angle in client_rotation_labels])

# Create confusion matrix
confusion_matrix = np.zeros((n_clusters, n_clusters), dtype=int)
for true_label, pred_label in zip(true_clusters, predicted_clusters):
    confusion_matrix[true_label, pred_label] += 1

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Pred {i}' for i in range(n_clusters)],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(n_clusters)])
plt.title('Confusion Matrix: True Rotation Clusters vs Predicted Clusters\n(Based on Weight Updates)')
plt.ylabel('True Rotation Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.show()

# Calculate purity for each cluster
print("\nCluster Purity Analysis:")
for pred_cluster in range(n_clusters):
    mask = predicted_clusters == pred_cluster
    if np.sum(mask) > 0:
        true_labels_in_cluster = true_clusters[mask]
        unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
        dominant_label = unique[np.argmax(counts)]
        purity = np.max(counts) / np.sum(mask)
        print(f"  Predicted Cluster {pred_cluster}:")
        print(f"    - Dominant rotation: {rotation_angles[dominant_label]}°")
        print(f"    - Purity: {purity:.2%}")
        print(f"    - Distribution: {dict(zip([rotation_angles[u] for u in unique], counts))}")

# Calculate overall accuracy (best matching)
from scipy.optimize import linear_sum_assignment

# Find best assignment between predicted and true clusters
cost_matrix = -confusion_matrix  # Negative for maximization
row_ind, col_ind = linear_sum_assignment(cost_matrix)

correctly_assigned = sum(confusion_matrix[i, j] for i, j in zip(row_ind, col_ind))
total_clients = len(true_clusters)
accuracy = correctly_assigned / total_clients

print(f"\nOverall Clustering Accuracy (best assignment): {accuracy:.2%}")
print(f"Correctly clustered clients: {correctly_assigned}/{total_clients}")

## Step 9: Visualize Weight Updates with PCA

Use PCA to reduce dimensionality and visualize the weight update space.

In [ ]:
# Apply PCA to reduce to 2D for visualization
pca = PCA(n_components=2, random_state=CONFIG['seed'])
updates_2d = pca.fit_transform(last_layer_updates)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Colored by true rotation cluster
ax1 = axes[0]
for rotation_idx, angle in enumerate(rotation_angles):
    mask = true_clusters == rotation_idx
    ax1.scatter(updates_2d[mask, 0], updates_2d[mask, 1], 
               label=f'{angle}°', alpha=0.6, s=100)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax1.set_title('Weight Updates in PCA Space (True Rotation Clusters)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Colored by predicted cluster
ax2 = axes[1]
colors = ['red', 'blue', 'green', 'orange']
for cluster_id in range(n_clusters):
    mask = predicted_clusters == cluster_id
    ax2.scatter(updates_2d[mask, 0], updates_2d[mask, 1], 
               label=f'Cluster {cluster_id}', alpha=0.6, s=100, 
               color=colors[cluster_id])
    
# Plot cluster centers
cluster_centers_2d = pca.transform(kmeans.cluster_centers_)
ax2.scatter(cluster_centers_2d[:, 0], cluster_centers_2d[:, 1], 
           marker='X', s=300, c='black', edgecolors='white', linewidths=2,
           label='Centroids', zorder=5)

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax2.set_title('Weight Updates in PCA Space (Predicted Clusters)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Analyze Update Magnitudes Across Clusters

Compare the magnitude of weight updates across different rotation clusters.

In [ ]:
# Calculate L2 norms of updates for each client
update_norms = np.linalg.norm(last_layer_updates, axis=1)

# Group by true rotation cluster
rotation_norms = {angle: [] for angle in rotation_angles}
for client_idx, angle in enumerate(client_rotation_labels):
    rotation_norms[angle].append(update_norms[client_idx])

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Box plot of update norms by rotation
ax1 = axes[0]
data_for_boxplot = [rotation_norms[angle] for angle in rotation_angles]
bp = ax1.boxplot(data_for_boxplot, labels=[f'{angle}°' for angle in rotation_angles],
                  patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax1.set_xlabel('Rotation Angle')
ax1.set_ylabel('L2 Norm of Weight Updates')
ax1.set_title('Distribution of Update Magnitudes by Rotation Cluster')
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Average update norms
ax2 = axes[1]
avg_norms = [np.mean(rotation_norms[angle]) for angle in rotation_angles]
std_norms = [np.std(rotation_norms[angle]) for angle in rotation_angles]
x_pos = np.arange(len(rotation_angles))
ax2.bar(x_pos, avg_norms, yerr=std_norms, capsize=5, alpha=0.7, color='steelblue')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'{angle}°' for angle in rotation_angles])
ax2.set_xlabel('Rotation Angle')
ax2.set_ylabel('Average L2 Norm of Updates')
ax2.set_title('Average Update Magnitude by Rotation Cluster')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print statistics
print("Update magnitude statistics by rotation:")
for angle in rotation_angles:
    norms = rotation_norms[angle]
    print(f"\n{angle}°:")
    print(f"  Mean: {np.mean(norms):.4f}")
    print(f"  Std:  {np.std(norms):.4f}")
    print(f"  Min:  {np.min(norms):.4f}")
    print(f"  Max:  {np.max(norms):.4f}")

## Step 11: Hierarchical Clustering Comparison

Compare K-Means with Hierarchical clustering using weight updates.

In [ ]:
# Perform hierarchical clustering
from sklearn.cluster import AgglomerativeClustering

hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
hierarchical_clusters = hierarchical.fit_predict(last_layer_updates)

# Calculate silhouette score for hierarchical clustering
hierarchical_silhouette = silhouette_score(last_layer_updates, hierarchical_clusters)

print(f"Hierarchical Clustering Results:")
print(f"Silhouette Score: {hierarchical_silhouette:.4f}")
print(f"\nCluster sizes:")
for i in range(n_clusters):
    count = np.sum(hierarchical_clusters == i)
    print(f"  Cluster {i}: {count} clients")

# Create confusion matrix for hierarchical clustering
hierarchical_confusion = np.zeros((n_clusters, n_clusters), dtype=int)
for true_label, pred_label in zip(true_clusters, hierarchical_clusters):
    hierarchical_confusion[true_label, pred_label] += 1

# Calculate accuracy for hierarchical clustering
cost_matrix_hier = -hierarchical_confusion
row_ind_hier, col_ind_hier = linear_sum_assignment(cost_matrix_hier)
correctly_assigned_hier = sum(hierarchical_confusion[i, j] for i, j in zip(row_ind_hier, col_ind_hier))
accuracy_hier = correctly_assigned_hier / total_clients

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means confusion matrix
ax1 = axes[0]
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=[f'Pred {i}' for i in range(n_clusters)],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(n_clusters)])
ax1.set_title(f'K-Means Clustering\nAccuracy: {accuracy:.2%}, Silhouette: {silhouette:.4f}')
ax1.set_ylabel('True Rotation Cluster')
ax1.set_xlabel('Predicted Cluster')

# Hierarchical confusion matrix
ax2 = axes[1]
sns.heatmap(hierarchical_confusion, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=[f'Pred {i}' for i in range(n_clusters)],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(n_clusters)])
ax2.set_title(f'Hierarchical Clustering\nAccuracy: {accuracy_hier:.2%}, Silhouette: {hierarchical_silhouette:.4f}')
ax2.set_ylabel('True Rotation Cluster')
ax2.set_xlabel('Predicted Cluster')

plt.tight_layout()
plt.show()

print(f"\nComparison Summary:")
print(f"K-Means:        Accuracy={accuracy:.2%}, Silhouette={silhouette:.4f}")
print(f"Hierarchical:   Accuracy={accuracy_hier:.2%}, Silhouette={hierarchical_silhouette:.4f}")

## Summary

This notebook demonstrated clustering clients based on **weight updates** instead of raw gradients:

**Key Findings:**
1. **Weight Updates**: Δw = w_after - w_before computed locally for each client
2. **Privacy**: Better than raw gradients as updates are what FL algorithms naturally compute
3. **Clustering Quality**: Measured by silhouette score and alignment with true rotation clusters
4. **Comparison**: K-Means vs Hierarchical clustering on the same updates

**Advantages over Raw Gradients:**
- More aligned with FedAvg and other FL algorithms
- Updates are already computed during local training
- Typically smaller magnitude than accumulated gradients
- Still captures client data heterogeneity

**Next Steps:**
- Consider differential privacy mechanisms for updates
- Explore secure aggregation protocols
- Test with different clustering algorithms (DBSCAN, spectral clustering)
- Analyze updates from multiple layers or full model